# Unificação das bases da Saúde de Criciúma

**Período dos pagamentos:** 2023, 2024 e 2025.

**Pergunta do projeto:** Como o perfil das despesas e a execução orçamentária se associam ao intervalo entre o empenho e o pagamento na Saúde de Criciúma, e como esses padrões podem apoiar a programação dos desembolsos?

Neste notebook vamos organizar e unir as bases e calcular o intervalo entre empenho e pagamento. Cada linha final representa um registro de pagamento. A transformação das categorias em 0/1 e a análise estatística ficam para a próxima etapa.

Execute as células na ordem. A pasta `dados` deve estar ao lado deste notebook. Os arquivos foram apenas copiados e renomeados para facilitar a leitura; seus conteúdos originais foram preservados.


## 1. Importar a biblioteca

Usaremos somente `pandas`, como nos exercícios. A união será feita com listas, comparações, `for` e `if`.


In [1]:
import pandas as pd


## 2. Informar os arquivos e escolher as colunas

Os CSVs da prefeitura contêm o registro principal e várias tabelas menores logo abaixo dele. Por isso, precisamos separar as linhas de orçamento e de pagamentos antes de unir as informações.

As três colunas da chave de união são: **entidade + exercício do empenho + número da despesa**. O número da despesa é obtido na seção `dotacaoOrcamentaria` de cada empenho.


In [2]:
arquivos_orcamento = [
    "dados/orcamento_2023.csv",
    "dados/orcamento_2024.csv",
    "dados/orcamento_2025.csv"
]

arquivos_execucao = [
    "dados/execucao_2023.csv",
    "dados/execucao_2024.csv",
    "dados/execucao_2025.csv"
]

colunas_empenho = [
    "Nome da entidade", "Ano de exercício", "Número do empenho",
    "Data do empenho", "Descrição da função", "Descrição da subfunção",
    "Programa", "Descrição do órgão", "Descrição da unidade do empenho",
    "Tipo de empenho", "Categoria do empenho", "Máscara do elemento",
    "Elemento", "Grupo do elemento", "Modalidade de aplicação",
    "Tipo de recurso", "Descrição do recurso", "Projeto/atividade",
    "Valor do empenho"
]

colunas_pagamento_no_csv = [
    "Nome da entidade", "Ano de cadastro", "Ano de exercício",
    "Data do empenho", "Número do empenho", "Código do elemento",
    "Descrição do órgão", "Histórico do empenho", "Tipo de empenho"
]

# Nas linhas da tabela interna, essas nove posições contêm os nove
# campos originais do pagamento, não os dados do empenho.
# Guardaremos os nove para comparar registros integralmente iguais.

colunas_valores_orcamento = [
    "Valor orçado da despesa R$", "Valor alterado da despesa R$",
    "Valor atualizado da despesa R$", "Empenhado atualizado da despesa R$",
    "Liquidado atualizado da despesa R$", "Saldo a liquidar da despesa R$",
    "Anulações do pagamento da despesa R$", "Valor pago atualizado da despesa R$",
    "Saldo a pagar da despesa - empenhado R$",
    "Saldo a pagar da despesa - liquidado R$",
    "% Empenhado sobre orçado", "% Empenhado sobre orçado atualizado",
    "% Liquidado sobre orçado", "% Liquidado sobre orçado atualizado",
    "% Pago sobre orçado", "% Pago sobre orçado atualizado"
]


## 3. Organizar os dados de orçamento

`dtype=str` mantém os códigos e datas como texto durante a organização. `keep_default_na=False` mantém os campos vazios como texto vazio. São parâmetros da mesma leitura `pd.read_csv` usada nas aulas.

O filtro pela função **Saúde** seleciona os registros principais e exclui os cabeçalhos e as tabelas internas. Linhas de orçamento completamente iguais serão guardadas uma única vez.


In [3]:
orcamentos = []
chaves_orcamento = []
valores_orcamento = []

for arquivo in arquivos_orcamento:
    tabela = pd.read_csv(arquivo, dtype=str, keep_default_na=False)

    for i in tabela.index:
        if tabela["Função"][i] == "Saúde":
            linha = []
            for coluna in tabela.columns:
                linha += [tabela[coluna][i]]

            if linha not in orcamentos:
                orcamentos += [linha]

                chave = [
                    tabela["Entidade"][i],
                    tabela["Ano"][i],
                    tabela["Código da despesa"][i]
                ]
                chaves_orcamento += [chave]

                valores = []
                for coluna in colunas_valores_orcamento:
                    valores += [tabela[coluna][i]]
                valores_orcamento += [valores]

print("Registros de orçamento da Saúde:", len(orcamentos))


Registros de orçamento da Saúde: 215


## 4. Separar os pagamentos e guardar os dados do empenho

Cada empenho possui suas próprias seções de pagamentos e dotação. Guardaremos os pagamentos até chegar ao próximo empenho; assim, o número da despesa já terá sido encontrado.

As datas originais estão no formato `AAAA-MM-DD`. A comparação de textos nesse formato permite selecionar os pagamentos de 2023 a 2025, sem calcular datas ainda.

Dentro do mesmo empenho, pagamentos iguais nos nove campos originais serão guardados uma única vez. Essa é a regra conservadora de limpeza utilizada no teste. Como o arquivo não traz um identificador único de transação preenchido, dois lançamentos realmente distintos e integralmente iguais não podem ser diferenciados; a regra deve constar na descrição da base.


In [4]:
pagamentos = []
pagamentos_repetidos = 0
empenhos_sem_numero_despesa = 0
empenhos_com_mais_de_uma_despesa = 0

for arquivo in arquivos_execucao:
    tabela = pd.read_csv(arquivo, dtype=str, keep_default_na=False)

    # A primeira linha do arquivo já foi utilizada como cabeçalho.
    proxima_linha_empenho = True
    proxima_linha_despesa = False
    lendo_pagamentos = False
    empenho_saude = False
    dados_empenho = []
    pagamentos_do_empenho = []
    numeros_despesa = []

    for i in tabela.index:
        primeiro_campo = tabela["Nome da entidade"][i]

        if primeiro_campo == "Nome da entidade":
            if empenho_saude and len(pagamentos_do_empenho) > 0:
                if len(numeros_despesa) == 1:
                    for pagamento in pagamentos_do_empenho:
                        pagamentos += [dados_empenho + [numeros_despesa[0]] + pagamento]
                elif len(numeros_despesa) == 0:
                    empenhos_sem_numero_despesa += 1
                else:
                    empenhos_com_mais_de_uma_despesa += 1

            proxima_linha_empenho = True
            lendo_pagamentos = False
            proxima_linha_despesa = False
            pagamentos_do_empenho = []
            numeros_despesa = []

        elif proxima_linha_empenho:
            empenho_saude = tabela["Descrição da função"][i] == "Saúde"
            dados_empenho = []
            if empenho_saude:
                for coluna in colunas_empenho:
                    dados_empenho += [tabela[coluna][i]]
            proxima_linha_empenho = False

        elif primeiro_campo == "- - - - -" or primeiro_campo == ">":
            lendo_pagamentos = False
            proxima_linha_despesa = False

        elif primeiro_campo == "Código da liquidação":
            lendo_pagamentos = True

        elif primeiro_campo == "UUID":
            proxima_linha_despesa = True
            lendo_pagamentos = False

        elif proxima_linha_despesa:
            if empenho_saude:
                numero_despesa = tabela["Ano de cadastro"][i]
                if numero_despesa not in numeros_despesa:
                    numeros_despesa += [numero_despesa]
            proxima_linha_despesa = False

        elif lendo_pagamentos and empenho_saude:
            data_pagamento = tabela["Ano de exercício"][i]
            if data_pagamento >= "2023-01-01" and data_pagamento < "2026-01-01":
                pagamento = []
                for coluna in colunas_pagamento_no_csv:
                    pagamento += [tabela[coluna][i]]

                if pagamento not in pagamentos_do_empenho:
                    pagamentos_do_empenho += [pagamento]
                else:
                    pagamentos_repetidos += 1

    # Guardar também o último empenho de cada arquivo.
    if empenho_saude and len(pagamentos_do_empenho) > 0:
        if len(numeros_despesa) == 1:
            for pagamento in pagamentos_do_empenho:
                pagamentos += [dados_empenho + [numeros_despesa[0]] + pagamento]
        elif len(numeros_despesa) == 0:
            empenhos_sem_numero_despesa += 1
        else:
            empenhos_com_mais_de_uma_despesa += 1

print("Pagamentos separados:", len(pagamentos))
print("Pagamentos integralmente repetidos retirados:", pagamentos_repetidos)
print("Empenhos com pagamento e sem número da despesa:", empenhos_sem_numero_despesa)
print("Empenhos com pagamento e mais de um número de despesa:", empenhos_com_mais_de_uma_despesa)


Pagamentos separados: 26125
Pagamentos integralmente repetidos retirados: 114
Empenhos com pagamento e sem número da despesa: 0
Empenhos com pagamento e mais de um número de despesa: 0


## 5. Unir pagamentos e orçamento

Para cada pagamento, procuraremos a mesma entidade, o mesmo exercício do empenho e o mesmo número da despesa. Só uniremos quando houver **uma única correspondência**.

O orçamento é uma característica da dotação: vários pagamentos podem compartilhar os mesmos valores orçamentários. Esses valores não devem ser somados novamente nas linhas da base unificada.


In [5]:
linhas_unificadas = []
pagamentos_sem_orcamento = 0
pagamentos_com_orcamento_ambiguo = 0

for pagamento in pagamentos:
    # Posições 0 e 1: entidade e exercício; posição 19: número da despesa.
    chave_pagamento = [pagamento[0], pagamento[1], pagamento[19]]
    correspondencias = 0
    valores_encontrados = []
    j = 0

    for chave in chaves_orcamento:
        if chave_pagamento == chave:
            correspondencias += 1
            valores_encontrados = valores_orcamento[j]
        j += 1

    if correspondencias == 1:
        # Depois dos 19 campos do empenho e do número da despesa,
        # as posições 22, 23 e 28 são data, tipo e valor do pagamento.
        linha = []
        k = 0
        for valor in pagamento:
            if k < 20:
                linha += [valor]
            k += 1

        linha += [pagamento[22], pagamento[23], pagamento[28]]
        linhas_unificadas += [linha + valores_encontrados]

    elif correspondencias == 0:
        pagamentos_sem_orcamento += 1
    else:
        pagamentos_com_orcamento_ambiguo += 1

colunas_unificadas = colunas_empenho + [
    "Número da despesa", "Data do pagamento", "Tipo de pagamento",
    "Valor do pagamento R$"
] + colunas_valores_orcamento

base_unificada = pd.DataFrame(linhas_unificadas, columns=colunas_unificadas)


## 6. Calcular o intervalo entre empenho e pagamento

Para fazer o cálculo solicitado, usaremos `pd.to_datetime` para converter as duas colunas de texto em datas. Depois, subtrairemos uma da outra. `.dt.days` apresenta o resultado em dias inteiros.

Essas duas operações de data foram acrescentadas para atender ao cálculo autorizado; elas não aparecem nos exercícios enviados. O restante continua usando os conceitos básicos das aulas.

A nova coluna será **intervalo_empenho_pagamento_dias**. Ela mede tempo decorrido, e não atraso, pois não temos a data de vencimento da obrigação.


In [6]:
base_unificada["Data do empenho"] = pd.to_datetime(
    base_unificada["Data do empenho"], format="%Y-%m-%d"
)

base_unificada["Data do pagamento"] = pd.to_datetime(
    base_unificada["Data do pagamento"], format="%Y-%m-%d"
)

base_unificada["intervalo_empenho_pagamento_dias"] = (
    base_unificada["Data do pagamento"] - base_unificada["Data do empenho"]
).dt.days


## 7. Conferir o resultado

Com os arquivos fornecidos, o esperado é **26.125 linhas**, **40 colunas**, nenhum pagamento sem orçamento e nenhuma correspondência ambígua. As duas datas foram convertidas, e os valores monetários mantêm a representação original. Ao ler o CSV entregue com `pd.read_csv`, as colunas de valores serão reconhecidas como numéricas.

Este notebook termina na base unificada com o intervalo calculado. Não há cálculo de correlação, normalização, modelo ou criação dos indicadores 0/1.


In [7]:
print("Linhas da base unificada:", len(base_unificada))
print("Colunas da base unificada:", len(base_unificada.columns))
print("Pagamentos sem orçamento:", pagamentos_sem_orcamento)
print("Pagamentos com mais de uma correspondência:", pagamentos_com_orcamento_ambiguo)

base_unificada.head()


Linhas da base unificada: 26125
Colunas da base unificada: 40
Pagamentos sem orçamento: 0
Pagamentos com mais de uma correspondência: 0


,Nome da entidade,Ano de exercício,Número do empenho,Data do empenho,Descrição da função,Descrição da subfunção,Programa,Descrição do órgão,Descrição da unidade do empenho,Tipo de empenho,Categoria do empenho,Máscara do elemento,Elemento,Grupo do elemento,Modalidade de aplicação,Tipo de recurso,Descrição do recurso,Projeto/atividade,Valor do empenho,Número da despesa,Data do pagamento,Tipo de pagamento,Valor do pagamento R$,Valor orçado da despesa R$,Valor alterado da despesa R$,Valor atualizado da despesa R$,Empenhado atualizado da despesa R$,Liquidado atualizado da despesa R$,Saldo a liquidar da despesa R$,Anulações do pagamento da despesa R$,Valor pago atualizado da despesa R$,Saldo a pagar da despesa - empenhado R$,Saldo a pagar da despesa - liquidado R$,% Empenhado sobre orçado,% Empenhado sobre orçado atualizado,% Liquidado sobre orçado,% Liquidado sobre orçado atualizado,% Pago sobre orçado,% Pago sobre orçado atualizado,intervalo_empenho_pagamento_dias
0,PREFEITURA MUNICIPAL DE CRICIUMA,2023,7295,2023-11-23,Saúde,Tecnologia da Informatização,SAUDE PARA TODOS,GABINETE DO PREFEITO,Gabinete do Prefeito,Ordinário,PROCESSO,44905235000000,Equipamentos de Processamento de Dados,INVESTIMENTOS,Aplicações Diretas,vinculado,Transferências de Estado referentes a Convênios e Instrumentos Congêneres vinculados à Saúde/ Superávit Exercício Anterior,Aquisição de equipamentos e materiais permanentes de TI,58514,229,2024-02-28,Débito em conta,36714.0,0,521444.4,521444.4,317314.0,21800.0,295514.0,0,261.6,317052.4,21538.4,0.0,60.85,0.0,4.18,0.0,199328.9,97
1,PREFEITURA MUNICIPAL DE CRICIUMA,2023,7295,2023-11-23,Saúde,Tecnologia da Informatização,SAUDE PARA TODOS,GABINETE DO PREFEITO,Gabinete do Prefeito,Ordinário,PROCESSO,44905235000000,Equipamentos de Processamento de Dados,INVESTIMENTOS,Aplicações Diretas,vinculado,Transferências de Estado referentes a Convênios e Instrumentos Congêneres vinculados à Saúde/ Superávit Exercício Anterior,Aquisição de equipamentos e materiais permanentes de TI,58514,229,2024-02-15,Débito em conta,21800.0,0,521444.4,521444.4,317314.0,21800.0,295514.0,0,261.6,317052.4,21538.4,0.0,60.85,0.0,4.18,0.0,199328.9,84
2,PREFEITURA MUNICIPAL DE CRICIUMA,2023,7279,2023-11-23,Saúde,Tecnologia da Informatização,SAUDE PARA TODOS,GABINETE DO PREFEITO,Gabinete do Prefeito,Ordinário,PROCESSO,44905235000000,Equipamentos de Processamento de Dados,INVESTIMENTOS,Aplicações Diretas,Ordinário,Receitas de Impostos e de Transferência de Impostos - Saúde,Aquisição de equipamentos e materiais permanentes de TI,10900,229,2024-02-28,Débito em conta,10900.0,0,521444.4,521444.4,317314.0,21800.0,295514.0,0,261.6,317052.4,21538.4,0.0,60.85,0.0,4.18,0.0,199328.9,97
3,PREFEITURA MUNICIPAL DE CRICIUMA,2023,7278,2023-11-23,Saúde,Tecnologia da Informatização,SAUDE PARA TODOS,GABINETE DO PREFEITO,Gabinete do Prefeito,Ordinário,PROCESSO,44905235000000,Equipamentos de Processamento de Dados,INVESTIMENTOS,Aplicações Diretas,Ordinário,Receitas de Impostos e de Transferência de Impostos - Saúde,Aquisição de equipamentos e materiais permanentes de TI,73902.43,229,2024-02-28,Débito em conta,73902.43,0,521444.4,521444.4,317314.0,21800.0,295514.0,0,261.6,317052.4,21538.4,0.0,60.85,0.0,4.18,0.0,199328.9,97
4,PREFEITURA MUNICIPAL DE CRICIUMA,2023,7277,2023-11-23,Saúde,Tecnologia da Informatização,SAUDE PARA TODOS,GABINETE DO PREFEITO,Gabinete do Prefeito,Ordinário,PROCESSO,44905235000000,Equipamentos de Processamento de Dados,INVESTIMENTOS,Aplicações Diretas,vinculado,Transferências de Estado referentes a Convênios e Instrumentos Congêneres vinculados à Saúde/ Superávit Exercício Anterior,Aquisição de equipamentos e materiais permanentes de TI,173997.57,229,2024-02-28,Débito em conta,173997.57,0,521444.4,521444.4,317314.0,21800.0,295514.0,0,261.6,317052.4,21538.4,0.0,60.85,0.0,4.18,0.0,199328.9,97
